## 1. Project Overview


CO3057 Computer Vision and Digital Image Preprocessing Project

## 2. Library Imports


In [1]:
import os
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import cv2
import cv2.aruco as aruco
import json
import glob

## 3. Global Constants and Configuration

In [2]:
DIRPATH = "./flyingarucov2"

ARUCO_DICT = aruco.getPredefinedDictionary(aruco.DICT_ARUCO_MIP_36h12)

## 4. Utility Functions

In [3]:
def load_data(img_id):
    img_path = os.path.join(DIRPATH, f"{img_id}.jpg")
    json_path = os.path.join(DIRPATH, f"{img_id}.json")

    img = cv2.imread(img_path)
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    return img, data

In [4]:
def convert_to_yolo_file():
    label_dir = os.path.join(DIRPATH, 'labels')
    if os.path.exists(label_dir) and len(os.listdir(label_dir)) > 0:
        print(f'{label_dir} already exists.')
        return
        
    os.makedirs(label_dir, exist_ok=True)

    json_files = glob.glob(os.path.join(DIRPATH, '*.json'))

    for json_file in json_files:
        # 1. Lấy file ảnh
        file_id = os.path.basename(json_file).replace('.json', '')
        img_path = os.path.join(DIRPATH, f'{file_id}.jpg')

        # 2. Lấy shape để normalize
        img = cv2.imread(img_path)
        height, width, _ = img.shape

        # 3. Đọc file json
        with open(json_file, 'r') as f:
            data = json.load(f)

        lines = []
        for marker in data['markers']:
            corner = marker['corners']

            # Normalize data
            xs = [c[0] for c in corner]
            ys = [c[1] for c in corner]
                
            x_min, x_max = min(xs), max(xs)
            y_min, y_max = min(ys), max(ys)

            x_mid_norm = ((x_min + x_max) / 2) / width
            y_mid_norm = ((y_min + y_max) / 2) / height

            w_norm = (x_max - x_min) / width
            h_norm = (y_max - y_min) / height

            line = f'0 {x_mid_norm:.6f} {y_mid_norm:.6f} {w_norm:.6f} {h_norm:.6f}'
            lines.append(line)

        # 4. Ghi file
        txt_file = os.path.join(label_dir, f'{file_id}.txt')
        with open(txt_file, 'w') as f:
            f.write('\n'.join(lines))

convert_to_yolo_file()

In [5]:
import shutil
import random

def split_dataset(src_dir, train_ratio=.8):
    base_dir = "dataset"
    sub_dirs = ["images/train", "images/val", "labels/train", "labels/val"]

    for sub in sub_dirs:
        try:
            os.makedirs(os.path.join(base_dir, sub), exist_ok=True)
        except FileExistsError:
            print(f"Folder images and labels already exist")
            return
        
    all_ids = [f.replace('.jpg', '') for f in os.listdir(src_dir) if f.endswith('.jpg')]
    random.seed(42)
    random.shuffle(all_ids)

    split_index = int(len(all_ids) * train_ratio)
    train_ids = all_ids[:split_index]
    val_ids = all_ids[split_index:]

    def move_files(ids, split_name):
        label_src_dir = os.path.join(src_dir, "labels")
        for id in ids:
            # Di chuyển ảnh
            img_src = os.path.join(src_dir, f"{id}.jpg")
            img_dst = os.path.join(base_dir, f"images/{split_name}/{id}.jpg")
            if os.path.exists(img_src):
                shutil.copy(img_src, img_dst)

            txt_src = os.path.join(label_src_dir, f"{id}.txt")
            txt_dst = os.path.join(base_dir, f"labels/{split_name}/{id}.txt")
            if os.path.exists(txt_src):
                shutil.copy(txt_src, txt_dst)

    move_files(train_ids, "train")
    move_files(val_ids, "val")

split_dataset(DIRPATH)


In [9]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

## 5. Core

In [ ]:
results = model.train(
    data="data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project="aruco_btl",
    name="aruco_v4",
    workers=0,
    exist_ok=True
)

New https://pypi.org/project/ultralytics/8.4.31 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.24  Python-3.13.7 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=aruco_v3, nbs=64, nms=Fals

## 6. Evaluation